# Nuremberg Feature Exploratory Data Analysis

Comprehensive analysis of **1,800 engineered features** for ~30K grid cells in Nuremberg.
Features are derived from Sentinel-2 (optical) and Sentinel-1 (SAR) satellite imagery across 2 years x 3 seasons.

**Feature groups:**
- **S2 Reflectance Bands** (B02-B12, B8A): 10 bands x 7 stats x 6 year-season = 420 features
- **Spectral Indices** (NDVI, NDWI, NDBI, EVI2, ...): 15 indices x 5 stats x 6 = 450 features
- **SAR Backscatter** (VV, VH, Cross-Ratio): 336 temporal/seasonal features
- **Texture & Tasseled Cap** (edge, laplacian, TC brightness/greenness/wetness): ~500 features
- **Coverage** (finite_frac): 60 data quality features

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import re

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# Find project root & data directory
NB_DIR = Path('.').resolve()
DATA_DIR = NB_DIR / 'data' if (NB_DIR / 'data' / 'nuremberg_meta.parquet').exists() else NB_DIR / 'data'

PROJECT = NB_DIR
while not (PROJECT / 'mlp').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

DIAG_DIR = PROJECT / 'mlp' / 'diagrams'
DIAG_DIR.mkdir(parents=True, exist_ok=True)

# ESA WorldCover class definitions
CLASS_NAMES = ['tree_cover', 'shrubland', 'grassland', 'cropland', 'built_up', 'bare_sparse', 'water']
CLASS_LABELS = ['Tree Cover', 'Shrubland', 'Grassland', 'Cropland', 'Built-up', 'Bare/Sparse', 'Water']
CLASS_COLORS = ['#2d6a4f', '#a3b18a', '#6a994e', '#f4a261', '#e76f51', '#d4a373', '#0096c7']

print('Notebook dir:', NB_DIR)
print('Data dir:', DATA_DIR)
print('Project root:', PROJECT)

## 1. Load Nuremberg Features & Labels

In [ ]:
# Load from split parquet files
meta = pd.read_parquet(DATA_DIR / 'nuremberg_meta.parquet')
parts = sorted(DATA_DIR.glob('nuremberg_features_part*.parquet'))
print(f'Loading {len(parts)} feature files...')

feat_dfs = [pd.read_parquet(p) for p in parts]
df = pd.concat([meta] + feat_dfs, axis=1)

# Assign dominant class
df['label'] = df[CLASS_NAMES].values.argmax(axis=1)
df['label_name'] = df['label'].map(dict(enumerate(CLASS_LABELS)))

# Feature columns only
meta_cols = {'cell_id', 'valid_fraction', 'low_valid_fraction'} | set(CLASS_NAMES) | {'label', 'label_name'}
feature_cols = [c for c in df.columns if c not in meta_cols]

print(f'Nuremberg: {len(df):,} grid cells x {len(feature_cols):,} features')
print(f'\nClass distribution:')
for i, (name, label) in enumerate(zip(CLASS_NAMES, CLASS_LABELS)):
    n = (df.label == i).sum()
    print(f'  {label:15s}: {n:6,} cells ({n/len(df)*100:5.1f}%)')

# Reusable plotting variables
classes = sorted(df.label.unique())
class_labels_used = [CLASS_LABELS[c] for c in classes]
colors_used = [CLASS_COLORS[c] for c in classes]

## 2. Feature Inventory

In [ ]:
def parse_feature(col):
    m = re.match(r'(SAR_\w+)_(temporal_std|temporal_cv|summer_winter)_(\d{4})$', col)
    if m:
        return m.group(1), m.group(2), m.group(3), 'temporal'
    m = re.match(r'([A-Za-z0-9_]+?)_(mean|std|min|max|q25|median|q75|finite_frac)_(\d{4})_(spring|summer|autumn|winter)$', col)
    if m:
        return m.groups()
    return None, None, None, None

parsed = pd.DataFrame([parse_feature(c) for c in feature_cols],
                      columns=['band', 'stat', 'year', 'season'], index=feature_cols)
parsed = parsed.dropna(subset=['band'])

S2_BANDS = {'B02','B03','B04','B05','B06','B07','B08','B8A','B11','B12'}
VEG_INDICES = {'NDVI','NDWI','NDBI','NDMI','NBR','BSI','EVI2','SAVI','GNDVI','NDRE1','NDRE2','NDTI','CRI1','IRECI','MNDWI'}
SAR_BANDS = {b for b in parsed.band.unique() if b.startswith('SAR')}
TEXTURE = {'edge','lap'} | {b for b in parsed.band.unique() if b.startswith('TC_')}

cats = {}
for idx, row in parsed.iterrows():
    if row.band in S2_BANDS:       cats[idx] = 'S2 Reflectance'
    elif row.band in VEG_INDICES:  cats[idx] = 'Spectral Index'
    elif row.band in SAR_BANDS:    cats[idx] = 'SAR Backscatter'
    elif row.band in TEXTURE:      cats[idx] = 'Texture/TC'
    elif row.stat == 'finite_frac':cats[idx] = 'Coverage'
    else:                          cats[idx] = 'Other'

parsed['category'] = pd.Series(cats)

print('Feature categories:')
for cat, grp in parsed.groupby('category'):
    bands = sorted(grp.band.unique())
    print(f'  {cat:20s}: {len(grp):4d} features  ({", ".join(bands[:8])}{"..." if len(bands) > 8 else ""})')
print(f'  {"TOTAL":20s}: {len(parsed):4d}')

## 3. Helper Functions

In [ ]:
def boxplot_grid(col_list, title_list, suptitle, save_name, ncols=5, figw_per=4.4, figh=5):
    """Generic per-class boxplot grid for a list of columns."""
    n = len(col_list)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(figw_per*ncols, figh*nrows))
    if n == 1:
        axes = np.array([axes])
    axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

    for ax, col, title in zip(axes_flat, col_list, title_list):
        if col not in df.columns:
            ax.set_visible(False)
            continue
        data = [df.loc[df.label == c, col].dropna().values for c in classes]
        data = [d for d in data if len(d) > 0]
        if not data:
            ax.set_visible(False)
            continue

        bp = ax.boxplot(data, patch_artist=True, showfliers=False, widths=0.65)
        for patch, co in zip(bp['boxes'], colors_used):
            patch.set_facecolor(co)
            patch.set_alpha(0.75)
        for med in bp['medians']:
            med.set_color('black')
            med.set_linewidth(1.5)
        ax.set_xticklabels(class_labels_used, rotation=45, ha='right', fontsize=6)
        ax.set_title(title, fontsize=10, fontweight='bold')
        ax.grid(True, alpha=0.2)

    for i in range(len(col_list), len(axes_flat)):
        axes_flat[i].set_visible(False)

    plt.suptitle(suptitle, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(DIAG_DIR / save_name, dpi=150, bbox_inches='tight')
    plt.show()

print('Helper functions defined.')

## 4. Data Quality: Missing Values & Coverage

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# NaN % per category
nan_by_cat = {}
for cat in parsed.category.unique():
    cat_cols = parsed[parsed.category == cat].index.tolist()
    nan_pct = df[cat_cols].isna().mean().mean() * 100
    nan_by_cat[cat] = nan_pct

ax = axes[0]
cats_sorted = sorted(nan_by_cat.items(), key=lambda x: -x[1])
ax.barh([c[0] for c in cats_sorted], [c[1] for c in cats_sorted], color='#8b5cf6', alpha=0.8)
ax.set_xlabel('Mean NaN %')
ax.set_title('NaN Percentage by Category', fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')

finite_cols = [c for c in feature_cols if 'finite_frac' in c]
if finite_cols:
    ax = axes[1]
    finite_means = df[finite_cols].mean()
    ax.hist(finite_means, bins=30, color='#06b6d4', alpha=0.8, edgecolor='white')
    ax.set_xlabel('Mean Finite Fraction')
    ax.set_ylabel('Count')
    ax.set_title('S2 Band Coverage (finite_frac)', fontweight='bold')
    ax.axvline(finite_means.mean(), color='red', ls='--', label=f'Mean: {finite_means.mean():.3f}')
    ax.legend()

ax = axes[2]
ax.hist(df['valid_fraction'], bins=50, color='#10b981', alpha=0.8, edgecolor='white')
ax.set_xlabel('Valid Fraction')
ax.set_ylabel('Cells')
ax.set_title('Cell Valid Fraction Distribution', fontweight='bold')
ax.axvline(df['valid_fraction'].mean(), color='red', ls='--', label=f'Mean: {df["valid_fraction"].mean():.3f}')
ax.legend()

plt.suptitle('Data Quality Overview - Nuremberg', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(DIAG_DIR / 'eda_data_quality.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df.label.value_counts().sort_index()
bars = axes[0].bar(range(len(counts)), counts.values, color=[CLASS_COLORS[i] for i in counts.index])
axes[0].set_xticks(range(len(counts)))
axes[0].set_xticklabels([CLASS_LABELS[i] for i in counts.index], rotation=30, ha='right')
axes[0].set_ylabel('Grid Cells')
axes[0].set_title('Dominant Class Distribution', fontweight='bold')
for bar, cnt in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{cnt:,}', ha='center', va='bottom', fontsize=8)
axes[0].grid(True, alpha=0.2, axis='y')

axes[1].pie(counts.values, labels=[CLASS_LABELS[i] for i in counts.index],
            colors=[CLASS_COLORS[i] for i in counts.index],
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 8})
axes[1].set_title('Class Proportions', fontweight='bold')

plt.suptitle(f'Nuremberg - {len(df):,} Grid Cells', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DIAG_DIR / 'eda_nuremberg_class_dist.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. S2 Reflectance Bands - All Statistics

Per-class distributions of all 10 Sentinel-2 reflectance bands across **all 7 statistics**
(mean, std, min, max, q25, median, q75) for summer 2021.

In [ ]:
# S2 Bands - mean
cols = [f'{b}_mean_2021_summer' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
titles = [f'{b}\n(mean)' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
boxplot_grid(cols, titles, 'S2 Reflectance Bands (mean, Summer 2021) - Per Class',
             'eda_s2_bands_mean.png', ncols=5, figh=4.5)

In [ ]:
# S2 Bands - std
cols = [f'{b}_std_2021_summer' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
titles = [f'{b}\n(std)' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
boxplot_grid(cols, titles, 'S2 Reflectance Bands (std, Summer 2021) - Per Class',
             'eda_s2_bands_std.png', ncols=5, figh=4.5)

In [ ]:
# S2 Bands - min
cols = [f'{b}_min_2021_summer' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
titles = [f'{b}\n(min)' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
boxplot_grid(cols, titles, 'S2 Reflectance Bands (min, Summer 2021) - Per Class',
             'eda_s2_bands_min.png', ncols=5, figh=4.5)

In [ ]:
# S2 Bands - q25
cols = [f'{b}_q25_2021_summer' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
titles = [f'{b}\n(q25)' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
boxplot_grid(cols, titles, 'S2 Reflectance Bands (q25, Summer 2021) - Per Class',
             'eda_s2_bands_q25.png', ncols=5, figh=4.5)

In [ ]:
# S2 Bands - median
cols = [f'{b}_median_2021_summer' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
titles = [f'{b}\n(median)' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
boxplot_grid(cols, titles, 'S2 Reflectance Bands (median, Summer 2021) - Per Class',
             'eda_s2_bands_median.png', ncols=5, figh=4.5)

In [ ]:
# S2 Bands - q75
cols = [f'{b}_q75_2021_summer' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
titles = [f'{b}\n(q75)' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
boxplot_grid(cols, titles, 'S2 Reflectance Bands (q75, Summer 2021) - Per Class',
             'eda_s2_bands_q75.png', ncols=5, figh=4.5)

In [ ]:
# S2 Bands - max
cols = [f'{b}_max_2021_summer' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
titles = [f'{b}\n(max)' for b in ['B02', 'B03', 'B04', 'B05', 'B06', 'B07', 'B08', 'B8A', 'B11', 'B12']]
boxplot_grid(cols, titles, 'S2 Reflectance Bands (max, Summer 2021) - Per Class',
             'eda_s2_bands_max.png', ncols=5, figh=4.5)

## 7. Spectral Indices - All Statistics

All 15 spectral indices across all available statistics for summer 2021.

In [ ]:
# Spectral Indices - mean
indices = ['NDVI', 'NDWI', 'NDBI', 'EVI2', 'NDMI', 'NBR', 'BSI', 'SAVI', 'GNDVI', 'MNDWI', 'NDRE1', 'NDRE2', 'CRI1', 'IRECI', 'NDTI']
cols = [f'{idx}_mean_2021_summer' for idx in indices]
avail_cols = [c for c in cols if c in df.columns]
avail_titles = [c.replace('_mean_2021_summer', '') for c in avail_cols]
if avail_cols:
    boxplot_grid(avail_cols, avail_titles,
                 'Spectral Indices (mean, Summer 2021) - Per Class',
                 'eda_indices_mean.png', ncols=5, figh=4.5)
else:
    print('No mean columns found for spectral indices')

In [ ]:
# Spectral Indices - std
indices = ['NDVI', 'NDWI', 'NDBI', 'EVI2', 'NDMI', 'NBR', 'BSI', 'SAVI', 'GNDVI', 'MNDWI', 'NDRE1', 'NDRE2', 'CRI1', 'IRECI', 'NDTI']
cols = [f'{idx}_std_2021_summer' for idx in indices]
avail_cols = [c for c in cols if c in df.columns]
avail_titles = [c.replace('_std_2021_summer', '') for c in avail_cols]
if avail_cols:
    boxplot_grid(avail_cols, avail_titles,
                 'Spectral Indices (std, Summer 2021) - Per Class',
                 'eda_indices_std.png', ncols=5, figh=4.5)
else:
    print('No std columns found for spectral indices')

In [ ]:
# Spectral Indices - min
indices = ['NDVI', 'NDWI', 'NDBI', 'EVI2', 'NDMI', 'NBR', 'BSI', 'SAVI', 'GNDVI', 'MNDWI', 'NDRE1', 'NDRE2', 'CRI1', 'IRECI', 'NDTI']
cols = [f'{idx}_min_2021_summer' for idx in indices]
avail_cols = [c for c in cols if c in df.columns]
avail_titles = [c.replace('_min_2021_summer', '') for c in avail_cols]
if avail_cols:
    boxplot_grid(avail_cols, avail_titles,
                 'Spectral Indices (min, Summer 2021) - Per Class',
                 'eda_indices_min.png', ncols=5, figh=4.5)
else:
    print('No min columns found for spectral indices')

In [ ]:
# Spectral Indices - q25
indices = ['NDVI', 'NDWI', 'NDBI', 'EVI2', 'NDMI', 'NBR', 'BSI', 'SAVI', 'GNDVI', 'MNDWI', 'NDRE1', 'NDRE2', 'CRI1', 'IRECI', 'NDTI']
cols = [f'{idx}_q25_2021_summer' for idx in indices]
avail_cols = [c for c in cols if c in df.columns]
avail_titles = [c.replace('_q25_2021_summer', '') for c in avail_cols]
if avail_cols:
    boxplot_grid(avail_cols, avail_titles,
                 'Spectral Indices (q25, Summer 2021) - Per Class',
                 'eda_indices_q25.png', ncols=5, figh=4.5)
else:
    print('No q25 columns found for spectral indices')

In [ ]:
# Spectral Indices - median
indices = ['NDVI', 'NDWI', 'NDBI', 'EVI2', 'NDMI', 'NBR', 'BSI', 'SAVI', 'GNDVI', 'MNDWI', 'NDRE1', 'NDRE2', 'CRI1', 'IRECI', 'NDTI']
cols = [f'{idx}_median_2021_summer' for idx in indices]
avail_cols = [c for c in cols if c in df.columns]
avail_titles = [c.replace('_median_2021_summer', '') for c in avail_cols]
if avail_cols:
    boxplot_grid(avail_cols, avail_titles,
                 'Spectral Indices (median, Summer 2021) - Per Class',
                 'eda_indices_median.png', ncols=5, figh=4.5)
else:
    print('No median columns found for spectral indices')

In [ ]:
# Spectral Indices - q75
indices = ['NDVI', 'NDWI', 'NDBI', 'EVI2', 'NDMI', 'NBR', 'BSI', 'SAVI', 'GNDVI', 'MNDWI', 'NDRE1', 'NDRE2', 'CRI1', 'IRECI', 'NDTI']
cols = [f'{idx}_q75_2021_summer' for idx in indices]
avail_cols = [c for c in cols if c in df.columns]
avail_titles = [c.replace('_q75_2021_summer', '') for c in avail_cols]
if avail_cols:
    boxplot_grid(avail_cols, avail_titles,
                 'Spectral Indices (q75, Summer 2021) - Per Class',
                 'eda_indices_q75.png', ncols=5, figh=4.5)
else:
    print('No q75 columns found for spectral indices')

In [ ]:
# Spectral Indices - max
indices = ['NDVI', 'NDWI', 'NDBI', 'EVI2', 'NDMI', 'NBR', 'BSI', 'SAVI', 'GNDVI', 'MNDWI', 'NDRE1', 'NDRE2', 'CRI1', 'IRECI', 'NDTI']
cols = [f'{idx}_max_2021_summer' for idx in indices]
avail_cols = [c for c in cols if c in df.columns]
avail_titles = [c.replace('_max_2021_summer', '') for c in avail_cols]
if avail_cols:
    boxplot_grid(avail_cols, avail_titles,
                 'Spectral Indices (max, Summer 2021) - Per Class',
                 'eda_indices_max.png', ncols=5, figh=4.5)
else:
    print('No max columns found for spectral indices')

## 8. Seasonal Phenology (NDVI across seasons)

How NDVI varies across spring/summer/autumn reveals seasonal crop cycles and deciduous vs evergreen patterns.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
seasons_2021 = ['spring', 'summer', 'autumn']
season_labels = ['Spring 2021', 'Summer 2021', 'Autumn 2021']

for ax, season, slabel in zip(axes, seasons_2021, season_labels):
    col = f'NDVI_mean_2021_{season}'
    if col not in df.columns:
        continue
    data = [df.loc[df.label == c, col].dropna().values for c in classes]
    data_f = [d for d in data if len(d) > 0]
    bp = ax.boxplot(data_f, patch_artist=True, showfliers=False, widths=0.65)
    for patch, co in zip(bp['boxes'], colors_used):
        patch.set_facecolor(co); patch.set_alpha(0.75)
    for med in bp['medians']:
        med.set_color('black'); med.set_linewidth(1.5)
    ax.set_xticklabels(class_labels_used, rotation=45, ha='right', fontsize=7)
    ax.set_title(slabel, fontsize=11, fontweight='bold')
    ax.set_ylabel('NDVI (mean)')
    ax.grid(True, alpha=0.2)

plt.suptitle('NDVI Seasonal Phenology - Nuremberg', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DIAG_DIR / 'eda_ndvi_seasonal.png', dpi=150, bbox_inches='tight')
plt.show()

# Seasonal line profiles
fig, ax = plt.subplots(figsize=(10, 5))
for c, name, color in zip(classes, class_labels_used, colors_used):
    means = []
    for season in seasons_2021:
        col = f'NDVI_mean_2021_{season}'
        if col in df.columns:
            means.append(df.loc[df.label == c, col].median())
    ax.plot(season_labels, means, '-o', color=color, label=name, linewidth=2, markersize=8)

ax.set_ylabel('Median NDVI')
ax.set_title('Seasonal NDVI Profiles by Class - Nuremberg', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(DIAG_DIR / 'eda_ndvi_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. SAR Backscatter (Sentinel-1) - All Statistics

Sentinel-1 SAR data is weather-independent (penetrates clouds).
- **VV**: co-polarized backscatter (sensitive to surface roughness)
- **VH**: cross-polarized backscatter (sensitive to volume scattering / vegetation)
- **Cross-Ratio (CR)**: VH/VV ratio, indicates depolarization

In [ ]:
# SAR - mean
cols = [f'{sf}_mean_2021_summer' for sf in ['SAR_VV', 'SAR_VH', 'SAR_CR']]
titles = ['VV (co-pol)', 'VH (cross-pol)', 'Cross-Ratio']
avail = [(c, t) for c, t in zip(cols, titles) if c in df.columns]
if avail:
    boxplot_grid([c for c,t in avail], [f'SAR {t} (mean)' for c,t in avail],
                 'SAR Backscatter (mean, Summer 2021) - Per Class',
                 'eda_sar_mean.png', ncols=3, figw_per=6)
else:
    print('No mean SAR columns found')

In [ ]:
# SAR - std
cols = [f'{sf}_std_2021_summer' for sf in ['SAR_VV', 'SAR_VH', 'SAR_CR']]
titles = ['VV (co-pol)', 'VH (cross-pol)', 'Cross-Ratio']
avail = [(c, t) for c, t in zip(cols, titles) if c in df.columns]
if avail:
    boxplot_grid([c for c,t in avail], [f'SAR {t} (std)' for c,t in avail],
                 'SAR Backscatter (std, Summer 2021) - Per Class',
                 'eda_sar_std.png', ncols=3, figw_per=6)
else:
    print('No std SAR columns found')

In [ ]:
# SAR - min
cols = [f'{sf}_min_2021_summer' for sf in ['SAR_VV', 'SAR_VH', 'SAR_CR']]
titles = ['VV (co-pol)', 'VH (cross-pol)', 'Cross-Ratio']
avail = [(c, t) for c, t in zip(cols, titles) if c in df.columns]
if avail:
    boxplot_grid([c for c,t in avail], [f'SAR {t} (min)' for c,t in avail],
                 'SAR Backscatter (min, Summer 2021) - Per Class',
                 'eda_sar_min.png', ncols=3, figw_per=6)
else:
    print('No min SAR columns found')

In [ ]:
# SAR - q25
cols = [f'{sf}_q25_2021_summer' for sf in ['SAR_VV', 'SAR_VH', 'SAR_CR']]
titles = ['VV (co-pol)', 'VH (cross-pol)', 'Cross-Ratio']
avail = [(c, t) for c, t in zip(cols, titles) if c in df.columns]
if avail:
    boxplot_grid([c for c,t in avail], [f'SAR {t} (q25)' for c,t in avail],
                 'SAR Backscatter (q25, Summer 2021) - Per Class',
                 'eda_sar_q25.png', ncols=3, figw_per=6)
else:
    print('No q25 SAR columns found')

In [ ]:
# SAR - median
cols = [f'{sf}_median_2021_summer' for sf in ['SAR_VV', 'SAR_VH', 'SAR_CR']]
titles = ['VV (co-pol)', 'VH (cross-pol)', 'Cross-Ratio']
avail = [(c, t) for c, t in zip(cols, titles) if c in df.columns]
if avail:
    boxplot_grid([c for c,t in avail], [f'SAR {t} (median)' for c,t in avail],
                 'SAR Backscatter (median, Summer 2021) - Per Class',
                 'eda_sar_median.png', ncols=3, figw_per=6)
else:
    print('No median SAR columns found')

In [ ]:
# SAR - q75
cols = [f'{sf}_q75_2021_summer' for sf in ['SAR_VV', 'SAR_VH', 'SAR_CR']]
titles = ['VV (co-pol)', 'VH (cross-pol)', 'Cross-Ratio']
avail = [(c, t) for c, t in zip(cols, titles) if c in df.columns]
if avail:
    boxplot_grid([c for c,t in avail], [f'SAR {t} (q75)' for c,t in avail],
                 'SAR Backscatter (q75, Summer 2021) - Per Class',
                 'eda_sar_q75.png', ncols=3, figw_per=6)
else:
    print('No q75 SAR columns found')

In [ ]:
# SAR - max
cols = [f'{sf}_max_2021_summer' for sf in ['SAR_VV', 'SAR_VH', 'SAR_CR']]
titles = ['VV (co-pol)', 'VH (cross-pol)', 'Cross-Ratio']
avail = [(c, t) for c, t in zip(cols, titles) if c in df.columns]
if avail:
    boxplot_grid([c for c,t in avail], [f'SAR {t} (max)' for c,t in avail],
                 'SAR Backscatter (max, Summer 2021) - Per Class',
                 'eda_sar_max.png', ncols=3, figw_per=6)
else:
    print('No max SAR columns found')

## 10. SAR Temporal Features

Temporal variability (std, CV) and summer-winter contrasts in SAR data.

In [ ]:
temporal_sar = {
    'SAR_VV_temporal_std': 'VV Temporal Std',
    'SAR_VH_temporal_std': 'VH Temporal Std',
    'SAR_CR_temporal_std': 'CR Temporal Std',
    'SAR_VV_temporal_cv': 'VV Temporal CV',
    'SAR_VV_summer_winter': 'VV Summer-Winter',
    'SAR_VH_summer_winter': 'VH Summer-Winter',
}

avail_sar = {}
for key, label in temporal_sar.items():
    for year in ['2021', '2020']:
        col = f'{key}_{year}'
        if col in df.columns:
            avail_sar[col] = f'{label} ({year})'
            break

if avail_sar:
    boxplot_grid(list(avail_sar.keys()), list(avail_sar.values()),
                 'SAR Temporal Features - Per Class',
                 'eda_sar_temporal.png', ncols=min(len(avail_sar), 6))
else:
    print('No temporal SAR features found')

## 11. Texture & Tasseled Cap Features - All Statistics

- **Edge features** capture spatial heterogeneity (urban areas are more textured)
- **Tasseled Cap** transforms: brightness (soil/urban), greenness (vegetation), wetness (moisture)

In [ ]:
# Texture & TC - mean
texture_cols = []
texture_titles = []

# Edge / Laplacian features
for base in ['edge', 'lap']:
    col = f'{base}_mean_2021_summer'
    if col in df.columns:
        texture_cols.append(col)
        texture_titles.append(f'{base.title()} (mean)')

# Tasseled Cap
for c in sorted(df.columns):
    if c.startswith('TC_') and c.endswith('_mean_2021_summer'):
        texture_cols.append(c)
        nice = c.replace('_mean_2021_summer', '').replace('TC_', 'TC ')
        texture_titles.append(f'{nice.title()} (mean)')

if texture_cols:
    boxplot_grid(texture_cols, texture_titles,
                 'Texture & TC (mean, Summer 2021) - Per Class',
                 'eda_texture_tc_mean.png', ncols=4, figh=5)
else:
    print('No mean texture/TC features found')

In [ ]:
# Texture & TC - std
texture_cols = []
texture_titles = []

# Edge / Laplacian features
for base in ['edge', 'lap']:
    col = f'{base}_std_2021_summer'
    if col in df.columns:
        texture_cols.append(col)
        texture_titles.append(f'{base.title()} (std)')

# Tasseled Cap
for c in sorted(df.columns):
    if c.startswith('TC_') and c.endswith('_std_2021_summer'):
        texture_cols.append(c)
        nice = c.replace('_std_2021_summer', '').replace('TC_', 'TC ')
        texture_titles.append(f'{nice.title()} (std)')

if texture_cols:
    boxplot_grid(texture_cols, texture_titles,
                 'Texture & TC (std, Summer 2021) - Per Class',
                 'eda_texture_tc_std.png', ncols=4, figh=5)
else:
    print('No std texture/TC features found')

In [ ]:
# Texture & TC - min
texture_cols = []
texture_titles = []

# Edge / Laplacian features
for base in ['edge', 'lap']:
    col = f'{base}_min_2021_summer'
    if col in df.columns:
        texture_cols.append(col)
        texture_titles.append(f'{base.title()} (min)')

# Tasseled Cap
for c in sorted(df.columns):
    if c.startswith('TC_') and c.endswith('_min_2021_summer'):
        texture_cols.append(c)
        nice = c.replace('_min_2021_summer', '').replace('TC_', 'TC ')
        texture_titles.append(f'{nice.title()} (min)')

if texture_cols:
    boxplot_grid(texture_cols, texture_titles,
                 'Texture & TC (min, Summer 2021) - Per Class',
                 'eda_texture_tc_min.png', ncols=4, figh=5)
else:
    print('No min texture/TC features found')

In [ ]:
# Texture & TC - q25
texture_cols = []
texture_titles = []

# Edge / Laplacian features
for base in ['edge', 'lap']:
    col = f'{base}_q25_2021_summer'
    if col in df.columns:
        texture_cols.append(col)
        texture_titles.append(f'{base.title()} (q25)')

# Tasseled Cap
for c in sorted(df.columns):
    if c.startswith('TC_') and c.endswith('_q25_2021_summer'):
        texture_cols.append(c)
        nice = c.replace('_q25_2021_summer', '').replace('TC_', 'TC ')
        texture_titles.append(f'{nice.title()} (q25)')

if texture_cols:
    boxplot_grid(texture_cols, texture_titles,
                 'Texture & TC (q25, Summer 2021) - Per Class',
                 'eda_texture_tc_q25.png', ncols=4, figh=5)
else:
    print('No q25 texture/TC features found')

In [ ]:
# Texture & TC - median
texture_cols = []
texture_titles = []

# Edge / Laplacian features
for base in ['edge', 'lap']:
    col = f'{base}_median_2021_summer'
    if col in df.columns:
        texture_cols.append(col)
        texture_titles.append(f'{base.title()} (median)')

# Tasseled Cap
for c in sorted(df.columns):
    if c.startswith('TC_') and c.endswith('_median_2021_summer'):
        texture_cols.append(c)
        nice = c.replace('_median_2021_summer', '').replace('TC_', 'TC ')
        texture_titles.append(f'{nice.title()} (median)')

if texture_cols:
    boxplot_grid(texture_cols, texture_titles,
                 'Texture & TC (median, Summer 2021) - Per Class',
                 'eda_texture_tc_median.png', ncols=4, figh=5)
else:
    print('No median texture/TC features found')

In [ ]:
# Texture & TC - q75
texture_cols = []
texture_titles = []

# Edge / Laplacian features
for base in ['edge', 'lap']:
    col = f'{base}_q75_2021_summer'
    if col in df.columns:
        texture_cols.append(col)
        texture_titles.append(f'{base.title()} (q75)')

# Tasseled Cap
for c in sorted(df.columns):
    if c.startswith('TC_') and c.endswith('_q75_2021_summer'):
        texture_cols.append(c)
        nice = c.replace('_q75_2021_summer', '').replace('TC_', 'TC ')
        texture_titles.append(f'{nice.title()} (q75)')

if texture_cols:
    boxplot_grid(texture_cols, texture_titles,
                 'Texture & TC (q75, Summer 2021) - Per Class',
                 'eda_texture_tc_q75.png', ncols=4, figh=5)
else:
    print('No q75 texture/TC features found')

In [ ]:
# Texture & TC - max
texture_cols = []
texture_titles = []

# Edge / Laplacian features
for base in ['edge', 'lap']:
    col = f'{base}_max_2021_summer'
    if col in df.columns:
        texture_cols.append(col)
        texture_titles.append(f'{base.title()} (max)')

# Tasseled Cap
for c in sorted(df.columns):
    if c.startswith('TC_') and c.endswith('_max_2021_summer'):
        texture_cols.append(c)
        nice = c.replace('_max_2021_summer', '').replace('TC_', 'TC ')
        texture_titles.append(f'{nice.title()} (max)')

if texture_cols:
    boxplot_grid(texture_cols, texture_titles,
                 'Texture & TC (max, Summer 2021) - Per Class',
                 'eda_texture_tc_max.png', ncols=4, figh=5)
else:
    print('No max texture/TC features found')

## 12. Feature Correlation Heatmap

Correlation between representative features (mean, summer 2021) to identify redundancy.

In [ ]:
rep_feats = []
for band in ['B04', 'B08', 'B11']:
    c = f'{band}_mean_2021_summer'
    if c in df.columns: rep_feats.append(c)
for idx in ['NDVI', 'NDWI', 'NDBI', 'EVI2', 'NDMI', 'NBR', 'BSI']:
    c = f'{idx}_mean_2021_summer'
    if c in df.columns: rep_feats.append(c)
for sar in ['SAR_VV', 'SAR_VH', 'SAR_CR']:
    c = f'{sar}_mean_2021_summer'
    if c in df.columns: rep_feats.append(c)

corr = df[rep_feats].corr()
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')

nice_labels = [c.replace('_mean_2021_summer', '').replace('SAR_', 'SAR ') for c in rep_feats]
ax.set_xticks(range(len(rep_feats)))
ax.set_yticks(range(len(rep_feats)))
ax.set_xticklabels(nice_labels, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(nice_labels, fontsize=9)

for i in range(len(rep_feats)):
    for j in range(len(rep_feats)):
        val = corr.values[i, j]
        color = 'white' if abs(val) > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

plt.colorbar(im, ax=ax, label='Pearson Correlation', shrink=0.8)
ax.set_title('Feature Correlation - Representative Features (Summer 2021)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DIAG_DIR / 'eda_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Year-over-Year Comparison (2020 vs 2021)

How consistent are features between the two input years?

In [ ]:
compare_indices = ['NDVI', 'NDWI', 'NDBI', 'EVI2']
fig, axes = plt.subplots(1, len(compare_indices), figsize=(5*len(compare_indices), 5))

for ax, idx in zip(axes, compare_indices):
    col_2020 = f'{idx}_mean_2020_summer'
    col_2021 = f'{idx}_mean_2021_summer'
    if col_2020 not in df.columns or col_2021 not in df.columns:
        continue
    for c, name, color in zip(classes, class_labels_used, colors_used):
        mask = df.label == c
        x = df.loc[mask, col_2020].values
        y = df.loc[mask, col_2021].values
        valid = np.isfinite(x) & np.isfinite(y)
        if valid.sum() > 100:
            ax.scatter(x[valid][::3], y[valid][::3], c=color, alpha=0.15, s=3, label=name)
    lims = [ax.get_xlim(), ax.get_ylim()]
    lo = min(lims[0][0], lims[1][0])
    hi = max(lims[0][1], lims[1][1])
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.5, lw=1)
    ax.set_xlabel(f'{idx} 2020'); ax.set_ylabel(f'{idx} 2021')
    ax.set_title(idx, fontsize=12, fontweight='bold')
    ax.set_aspect('equal'); ax.grid(True, alpha=0.2)

handles = [mpatches.Patch(color=c, label=n) for c, n in zip(colors_used, class_labels_used)]
axes[-1].legend(handles=handles, fontsize=7, loc='lower right')
plt.suptitle('Year-over-Year Feature Consistency (Summer 2020 vs 2021)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(DIAG_DIR / 'eda_year_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Summary Statistics Table

In [ ]:
summary_feats = ['NDVI_mean_2021_summer', 'NDWI_mean_2021_summer', 'NDBI_mean_2021_summer',
                 'EVI2_mean_2021_summer', 'B04_mean_2021_summer', 'B08_mean_2021_summer',
                 'B11_mean_2021_summer', 'SAR_VV_mean_2021_summer', 'SAR_VH_mean_2021_summer']
summary_feats = [f for f in summary_feats if f in df.columns]

summary = df.groupby('label_name')[summary_feats].mean()
nice_cols = [c.replace('_mean_2021_summer', '').replace('SAR_', 'SAR ') for c in summary_feats]
summary.columns = nice_cols

print('Per-class mean feature values (Summer 2021):')
display(summary.round(4).style.background_gradient(cmap='YlOrRd', axis=0))

## Key Findings

1. **NDVI** is the strongest class discriminator - tree cover has the highest NDVI (~0.8), followed by grassland (~0.75); water is near zero (~0.2), built-up is low (~0.35)
2. **NDBI** (built-up index) separates urban and bare surfaces (positive values) from vegetated classes (negative values), though with overlapping distributions
3. **SAR VV/VH** shows urban double-bounce scattering (built-up has the highest backscatter at ~0.17 VV), while water has the lowest due to specular reflection (~0.07 VV). Cross-ratio is highest for water, indicating strong depolarization
4. **Seasonal NDVI profiles** show all vegetation classes increasing from spring to summer; cropland shows the largest seasonal swing (0.48 -> 0.55), while tree cover and grassland remain relatively stable at high values
5. **Feature redundancy is extreme**: NDBI-BSI correlation = 0.99, NDMI-NBR = 0.99, NDVI-EVI2 = 0.93. Many spectral indices carry near-identical information, though the model may still benefit from this for robustness
6. **Year consistency**: features are generally stable between 2020-2021, confirming temporal consistency of the input data
7. **Class imbalance in Nuremberg**: tree cover dominates (47.9%, 14.4K cells), followed by built-up (33.2%, 9.9K cells); bare/sparse (28 cells, 0.1%) and water (247 cells, 0.8%) are extreme minority classes